# KPSS (Kwiatkowski–Phillips–Schmidt–Shin) Test in Python

A walkthrough using `statsmodels` to perform the KPSS test: printing the test statistic, p-value, critical values, and automatically making the stationarity decision.

**Key idea:** KPSS's null hypothesis is the *opposite* of ADF's.

- **ADF** &nbsp;H0: the series has a unit root (non-stationary)
- **KPSS** H0: the series is stationary

Because of this, ADF and KPSS are complementary tests — running both together gives a much more informative picture than either alone.

## The `kpss_test` Helper Function

This function runs the KPSS test on a series and prints a full report, including the hypotheses, decision, and critical values.

In [ ]:
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import kpss


def kpss_test(series, regression="c", significance=0.05, name="Time Series"):
    """
    Perform the KPSS test and make a statistical decision.

    regression="c"  -> test level stationarity
    regression="ct" -> test trend stationarity
    """

    # Remove missing values
    series = pd.Series(series).dropna()

    # Run KPSS test
    result = kpss(series, regression=regression, nlags="auto")

    kpss_statistic = result[0]
    p_value = result[1]
    used_lags = result[2]
    critical_values = result[3]

    print("=" * 60)
    print(f"KPSS TEST: {name}")
    print("=" * 60)

    print(f"KPSS Statistic : {kpss_statistic:.4f}")
    print(f"p-value        : {p_value:.4f}")
    print(f"Used Lags      : {used_lags}")
    print(f"Regression     : {regression}")

    print("\nCritical Values:")
    for level, value in critical_values.items():
        print(f"  {level:>4} : {value:.4f}")

    print(f"\nSignificance Level (\u03b1): {significance}")

    # Decision using p-value
    print("\nDecision:")

    if p_value <= significance:
        print("Reject H0")
        print("Conclusion: The series is non-stationary.")
    else:
        print("Fail to Reject H0")
        print("Conclusion: The series is stationary.")

    print("\nHypotheses:")
    print("H0: The series is stationary.")
    print("H1: The series is non-stationary.")

    print("=" * 60)

    return {
        "kpss_statistic": kpss_statistic,
        "p_value": p_value,
        "critical_values": critical_values,
        "used_lags": used_lags,
        "stationary": p_value > significance
    }

## Example 1 — A Stationary Series

Generate white-noise-like data. Since white noise fluctuates around a constant mean with constant variance, we expect KPSS to **fail to reject H0** (evidence of stationarity).

In [ ]:
np.random.seed(42)

# White-noise-like stationary series
stationary_data = np.random.normal(
    loc=0,
    scale=1,
    size=500
)

result = kpss_test(
    stationary_data,
    regression="c",
    significance=0.05,
    name="Stationary Series"
)

Expected decision:

```
Fail to Reject H0
Conclusion: The series is stationary.
```

## Example 2 — A Non-Stationary Random Walk

Generate a random walk by cumulatively summing normal noise. Random walks are the classic example of a non-stationary process, so we expect KPSS to **reject H0**.

In [ ]:
np.random.seed(42)

# Random walk
random_walk = np.cumsum(
    np.random.normal(
        loc=0,
        scale=1,
        size=500
    )
)

result = kpss_test(
    random_walk,
    regression="c",
    significance=0.05,
    name="Random Walk"
)

Typical result:

```
Reject H0
Conclusion: The series is non-stationary.
```

Note: `statsmodels` may print an `InterpolationWarning` when the statistic falls outside the table range — this is expected for a strongly non-stationary series like a random walk, and doesn't affect the conclusion.

## Applying It to Stock Data

Load a CSV containing a `Close` price column and run the KPSS test on the raw closing price.

In [ ]:
df = pd.read_csv("stock_data.csv")

result = kpss_test(
    df["Close"],
    regression="c",
    significance=0.05,
    name="Stock Closing Price"
)

You might get output like:

```
============================================================
KPSS TEST: Stock Closing Price
============================================================
KPSS Statistic : 1.4210
p-value        : 0.0100
Used Lags      : 12
Regression     : c

Critical Values:
   10% : 0.3470
    5% : 0.4630
  2.5% : 0.5740
    1% : 0.7390

Significance Level (α): 0.05

Decision:
Reject H0
Conclusion: The series is non-stationary.

Hypotheses:
H0: The series is stationary.
H1: The series is non-stationary.
============================================================
```

Raw stock prices are typically non-stationary — this is expected, and agrees with what ADF would tell us on the same series.

## Then Test Returns

Prices are usually non-stationary, but **returns** (percentage changes) frequently are stationary. Let's compute returns and re-run the test.

In [ ]:
df["Return"] = df["Close"].pct_change()

result = kpss_test(
    df["Return"],
    regression="c",
    significance=0.05,
    name="Stock Returns"
)

You may obtain:

```
KPSS Statistic : 0.1050
p-value        : 0.1000

Decision:
Fail to Reject H0
Conclusion: The series is stationary.
```

This agrees with ADF's conclusion on returns — exactly the kind of cross-check that makes the two tests complementary.

## Comparing ADF and KPSS Side by Side

The most informative workflow runs **both** tests on the same series and cross-checks the conclusions.

| ADF | KPSS | Interpretation |
|---|---|---|
| Reject H0 | Fail to reject H0 | ✅ Strong evidence for stationarity |
| Fail to reject H0 | Reject H0 | ❌ Strong evidence for non-stationarity |
| Reject H0 | Reject H0 | ⚠️ Conflicting evidence — investigate further |
| Fail to reject H0 | Fail to reject H0 | ⚠️ Inconclusive — investigate further |

In [ ]:
from statsmodels.tsa.stattools import adfuller

def adf_quick(series, significance=0.05):
    series = pd.Series(series).dropna()
    stat, p_value, *_ = adfuller(series, autolag="AIC")
    decision = "Reject H0 (stationary)" if p_value <= significance else "Fail to reject H0 (non-stationary)"
    return p_value, decision


def compare_adf_kpss(series, name="Series", significance=0.05):
    adf_p, adf_decision = adf_quick(series, significance)
    kpss_result = kpss_test(series, regression="c", significance=significance, name=name)
    kpss_p = kpss_result["p_value"]
    kpss_decision = "Reject H0 (non-stationary)" if kpss_p <= significance else "Fail to reject H0 (stationary)"

    print(f"\n--- Summary for {name} ---")
    print(f"ADF  p-value = {adf_p:.4f} -> {adf_decision}")
    print(f"KPSS p-value = {kpss_p:.4f} -> {kpss_decision}")


# Example: compare on the stationary series generated above
compare_adf_kpss(stationary_data, name="Stationary Series")

## The Workflow

```
             TIME SERIES
                  |
                  v
             ADF TEST
                  |
                  v
             KPSS TEST
                  |
          +-------+--------+
          |                |
     Both indicate    Both indicate
      stationary      non-stationary
          |                |
          v                v
      Continue          Difference /
       modeling          transform
                             |
                             v
                         ADF + KPSS
                             |
                             v
                        Check again
```

## The Key Takeaway

ADF and KPSS are **complementary tests, not competing tests**.

| | ADF | KPSS |
|---|---|---|
| H0 | Non-stationary | Stationary |
| H1 | Stationary | Non-stationary |
| p ≤ 0.05 | Reject non-stationarity | Reject stationarity |

> For practical time-series work, running ADF + KPSS together is much more informative than relying on either test alone.